# Day 030 — Exercise 4: ai_pipeline_summary

**What you'll build:** `ai_pipeline_summary(step_results, model='llama3.2') -> str` — formats step results as a checkmark/cross text block and asks Ollama to summarise the pipeline run in 2–3 sentences.

**Why it matters:** A pipeline monitor that shows raw JSON is hard to read. A natural-language summary — 'The pipeline completed successfully in 0.3s' or 'Step process_content failed with a KeyError; downstream steps were skipped' — is immediately actionable for a human operator.

In [ ]:
import ollama

## Provided: run_step, chain_steps, summarize_run

In [ ]:
import time


def run_step(name: str, fn) -> dict:
    start = time.time()
    try:
        result = fn()
        return {
            "name":       name,
            "status":     "ok",
            "result":     result,
            "error":      None,
            "duration_s": round(time.time() - start, 3),
        }
    except Exception as e:
        return {
            "name":       name,
            "status":     "error",
            "result":     None,
            "error":      str(e),
            "duration_s": round(time.time() - start, 3),
        }


def chain_steps(steps: list, stop_on_error: bool = True) -> list:
    results = []
    failed  = False
    for name, fn in steps:
        if failed and stop_on_error:
            results.append({
                "name":       name,
                "status":     "skipped",
                "result":     None,
                "error":      None,
                "duration_s": 0.0,
            })
        else:
            step_result = run_step(name, fn)
            results.append(step_result)
            if step_result["status"] == "error":
                failed = True
    return results


def summarize_run(step_results: list) -> dict:
    statuses = [s["status"] for s in step_results]
    return {
        "total":            len(step_results),
        "passed":           statuses.count("ok"),
        "failed":           statuses.count("error"),
        "skipped":          statuses.count("skipped"),
        "total_duration_s": round(
            sum(s.get("duration_s", 0.0) for s in step_results), 3
        ),
        "all_ok":           all(s == "ok" for s in statuses),
    }

## Your Implementation

In [ ]:
def ai_pipeline_summary(
    step_results: list,
    model: str = 'llama3.2',
) -> str:
    """
    Return a 2–3 sentence natural-language summary of a pipeline run.

    Args:
        step_results: List of step result dicts from chain_steps.
        model:        Ollama model name.

    Returns:
        A concise string describing what happened in the pipeline.
    """
    # TODO: summary = summarize_run(step_results)
    # TODO: for each step: '  \u2713 name (dur)' or '  \u2717 name: error' or '  - name: skipped'
    # TODO: run_text = f'{passed}/{total} steps passed, {dur}s total\n' + '\n'.join(lines)
    # TODO: ollama.chat with system 'Summarise run in 2-3 sentences'
    # TODO: return response['message']['content']
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    def _sr(name, status, dur=0.05, error=None):
        return {'name': name, 'status': status, 'result': None,
                'error': error, 'duration_s': dur}

    ALL_OK   = [_sr('fetch', 'ok', 0.1), _sr('process', 'ok', 0.2), _sr('report', 'ok', 0.05)]
    WITH_ERR = [_sr('fetch', 'ok', 0.1), _sr('process', 'error', 0.02, 'KeyError: rows'),
                _sr('report', 'skipped', 0.0)]

    # Check 1: defined
    try:
        assert 'ai_pipeline_summary' in globals()
        passed += 1; print('\u2705 Check 1: ai_pipeline_summary defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    ok_report = None

    # Check 2: returns a string for all-ok input
    try:
        ok_report = ai_pipeline_summary(ALL_OK)
        assert isinstance(ok_report, str), \
            f'expected str, got {type(ok_report)}'
        passed += 1; print('\u2705 Check 2: returns a string')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: non-empty summary
    try:
        assert ok_report is not None
        assert len(ok_report.strip()) > 10, \
            f'summary too short: {ok_report!r}'
        passed += 1; print(f'\u2705 Check 3: summary is {len(ok_report)} chars')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: works with failed/skipped steps
    try:
        err_report = ai_pipeline_summary(WITH_ERR)
        assert isinstance(err_report, str) and len(err_report) > 10
        passed += 1; print('\u2705 Check 4: works with error/skipped steps')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: single-step pipeline also works
    try:
        one = [_sr('solo', 'ok', 0.01)]
        report = ai_pipeline_summary(one)
        assert isinstance(report, str) and len(report) > 5
        passed += 1; print('\u2705 Check 5: single-step pipeline summary works')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def ai_pipeline_summary(step_results: list, model: str = "llama3.2") -> str:
    summary = summarize_run(step_results)
    lines = []
    for s in step_results:
        if s["status"] == "ok":
            lines.append(f"  \u2713 {s['name']} ({s['duration_s']:.3f}s)")
        elif s["status"] == "error":
            lines.append(f"  \u2717 {s['name']}: {s['error']}")
        else:
            lines.append(f"  - {s['name']}: skipped")
    run_text = (
        f"{summary['passed']}/{summary['total']} steps passed, "
        f"{summary['total_duration_s']}s total\n"
        + "\n".join(lines)
    )
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a pipeline monitor. "
                    "Summarise a workflow run in 2\u20133 sentences. Be concise."
                ),
            },
            {
                "role": "user",
                "content": f"{run_text}\n\nSummarise the run:",
            },
        ],
    )
    return response["message"]["content"]
```

</details>